# Basic figures for 311 service requests vs EMS heat calls

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:

import os
import shii
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import datetime as dt

APP_TOKEN = os.getenv('NYC_OPEN_DATA_APP_TOKEN')

## 1. Prep data

In [ ]:

all_311_df = shii.prepare_all_311_requests(APP_TOKEN, output_path='./311_calls.gpkg')
heat_inc_df = shii.prepare_ems_calls(APP_TOKEN, output_path='./ems_calls.csv')

In [ ]:
#  Separate types of vent complaints
# all_311_df.loc[all_311_df['request_type']=='ventilation'].groupby('agency_name').size()
# all_311_df.loc[(all_311_df['request_type']=='ventilation') & (all_311_df['agency_name']=='Department of Health and Mental Hygiene'), 'request_type'] = 'vent_doh'
# all_311_df.loc[(all_311_df['request_type']=='ventilation') & (all_311_df['agency_name']=='Department of Housing Preservation and Development'), 'request_type'] = 'vent_house'

In [ ]:
full_df = shii.preprocess_merge_df(heat_inc_df, all_311_df, summer_only=False)

In [ ]:
# Catch-all category 
full_df['hydrant_all'] = full_df['hydrant'] + full_df['fhe']
# Normalize variables by population
# for col in ['ac','hydrant','power','ventilation','fhe', 'heat_ems_count']:
for col in ['ac','hydrant','hydrant_all','power','ventilation', 'fhe', 'heat_ems_count', 'tree']:
    full_df[col+'_norm'] = 100000*full_df[col]/full_df['population']

In [ ]:
# Group by date
date_grouped = full_df.groupby('date').sum()
date_grouped_mean = full_df.groupby('date').mean()
# Some things shouldn't be summed
per_cdta_vars = ['HVI_RANK', 'SURFACE_TEMP', 'MEDIAN_INCOME',
                 'GREENSPACE', 'PCT_HOUSEHOLDS_AC','PCT_BLACK_POP',
                 'population',
                 'temp', 'tmin','tmax', 'rhum', 'prcp',
                 'snwd', 'wspd', 'wpgt', 'pres', 'cldc']
date_grouped[per_cdta_vars] = date_grouped_mean[per_cdta_vars]

In [ ]:
full_df.reset_index()['date'].dt.day_of_week

In [ ]:
full_df.reset_index().groupby(full_df.reset_index()['date'].dt.day_of_week)['hydrant_all'].sum()

In [ ]:
# After 2022
post_2022_df = full_df.loc[full_df.index.get_level_values(1).year >= 2015]
cdta_grouped = post_2022_df.groupby('cdta').mean()
cdta_grouped_sum = post_2022_df.groupby('cdta').sum()
# Replace sums with mean values in cases where means don't make sense
per_cdta_vars = ['HVI_RANK', 'SURFACE_TEMP', 'MEDIAN_INCOME',
                 'GREENSPACE', 'PCT_HOUSEHOLDS_AC','PCT_BLACK_POP',
                 'population',]
cdta_grouped_sum[per_cdta_vars] = cdta_grouped[per_cdta_vars]


### Time series

In [ ]:

# Fig 2: Time series comparison
fig, axs = plt.subplots(2, 3, figsize=(12, 7))

start_date = '2024-06-01'
end_date = '2024-09-30'

color_list = ['darkred','darkorange', '#D62828', '#FCBF49', 'purple',  'green']
title_list = ['Daily Max Temperature', 'EMS Heat Calls', '311 Hydrant', '311 Ventilation', '311 Power',  '311 Tree']
col_list = ['tmax', 'heat_ems_count_norm', 'hydrant_all_norm',  'ventilation_norm', 'power_norm', 'tree_norm']
ax_list = axs.flatten()

for i, (col, ax, color, title) in enumerate(zip(col_list, ax_list, color_list, title_list)):
    if i==0:
        date_grouped['tmax'].plot(
            ax=ax,
            xlim=(start_date, end_date),
            ylim=(0,40),
            color='darkred',
            ylabel='Temperature (C)',
            xlabel='',
            title='Daily Max Temperature',
            lw=2
        )
    else:
        date_grouped.loc[pd.date_range(start_date, end_date)][col].plot(
            ax=ax,
            color=color,
            xlabel='',
            ylabel='Per 100k people',
            title=title,
            lw=2
        )

# Add heat event lines
heat_dates = ['2024-06-21', '2024-07-06', '2024-07-16', '2024-08-02', '2024-08-28']
for ax in ax_list:
    for hd in heat_dates:
        leg_handle = ax.axvline(hd, alpha=0.5, color='black', linestyle='--', lw=0.9)

ax_list[0].legend([leg_handle], ['Heat EMS Local Maxima'], loc='upper left')
plt.tight_layout()

In [ ]:

# Fig 2: Time series comparison
fig = plt.figure(figsize=(12, 9))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

start_date = '2024-05-01'
end_date = '2024-09-30'

# Max Temp across top (spans all 3 columns)
ax_temp = fig.add_subplot(gs[0, :])
date_grouped['tmax'].plot(
    xlim=(start_date, end_date),
    ylim=(0,40),
    ax=ax_temp,
    color='darkred',
    ylabel='Temperature (C)',
    xlabel='',
    title='Daily Max Temperature',
    lw=2
)

# 2x3 grid below
ax_list = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
    fig.add_subplot(gs[2, 0]),
    fig.add_subplot(gs[2, 1]),
    fig.add_subplot(gs[2, 2])
]

color_list = ['darkorange', '#D62828', '#FCBF49', 'purple', '#2A9D8F', 'green']
title_list = ['EMS Heat Calls', '311 Hydrant', '311 Ventilation', '311 Power', '311 AC', '311 Tree']
col_list = ['heat_ems_count_norm', 'hydrant_norm', 'ventilation_norm', 'power_norm', 'ac_norm', 'tree_norm']

for i, (col, ax, color, title) in enumerate(zip(col_list, ax_list, color_list, title_list)):
    date_grouped.loc[pd.date_range(start_date, end_date)][col].plot(
        ax=ax,
        color=color,
        xlabel='',
        ylabel='Per 100k people',
        title=title,
        lw=2
    )

# Add heat event lines
heat_dates = ['2024-06-21', '2024-07-06', '2024-07-16', '2024-08-02', '2024-08-28']
for ax in [ax_temp] + ax_list:
    for hd in heat_dates:
        leg_handle = ax.axvline(hd, alpha=0.5, color='black', linestyle='--', lw=0.9)

ax_temp.legend([leg_handle], ['Heat EMS Local Maxima'], loc='upper left')
plt.tight_layout()

### Maps

### Scatter plots

In [ ]:
cdta_map = shii.download_community_districts().rename(columns={'boro_cd':'cdta'}).set_index('cdta')[['geometry']]

In [ ]:
date_grouped['heat_ems_count_norm'].sum()

In [ ]:
import matplotlib

In [ ]:
# Fig 1: Heat EMS times series and map
# cdta_grouped_all = full_df.groupby('cdta').mean()
# cdta_grouped_sum_all = full_df.groupby('cdta').sum()
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
date_grouped['heat_ems_count'].loc[date_grouped.index.year>=2015].plot(
    ax=axs[0],
    xlabel='',
    ylabel='EMS Calls per day',
    title='Heat Exhaustion EMS Calls per day',
    color='darkorange'
    )


cdta_map.join(cdta_grouped_sum).plot(
    column='heat_ems_count_norm',
    legend=True,
    ax=axs[1],
    cmap='OrRd',
    norm=matplotlib.colors.LogNorm(vmin=10,
                                   vmax=1000)#cdta_grouped_sum_all.heat_ems_count_norm.max())
)
axs[1].set_title('Total Heat Exhaustion EMS Calls\n2015-2025, per 100k population')
axs[1].set_xlabel('Lon')
axs[1].set_ylabel('Lat')
fig.show()

# Real-time HVI

In [ ]:
# Get last 3 days of EMS calls and 311 requests
rolling_days = 3
rolling_columns = ['heat_ems_count','heat_ems_count_norm','ac','ac_norm','fhe','fhe_norm',
                   'hydrant','hydrant_norm','hydrant_all','hydrant_all_norm','power','power_norm',
                   'ventilation','ventilation_norm', 'tree_norm', 'tree']
heat_rolling = shii.compute_rolling(full_df, rolling_columns,window=dt.timedelta(days=rolling_days))
heat_rolling = heat_rolling.reset_index().set_index(['cdta','date'])
roll_df = full_df.join(heat_rolling, rsuffix='_last3')#.reset_index()

In [ ]:

roll_df.loc[roll_df.index.get_level_values(1).month.isin([6, 7,8]),
            ['ventilation_norm_last3', 'ac_norm_last3','power_norm_last3','heat_ems_count_norm_last3','hydrant_all_norm_last3', 'tree_norm_last3']].quantile(0.8)

In [ ]:
# Add flag thresholds
def compute_shii(df):
    threshold_dict = {
        'hydrant_all_norm_last3': 8.6,
        'ventilation_norm_last3': .8,
        # 'ac_norm_last3':0.0,
        # 'heat_ems_count_norm_last3':0.5,
        'power_norm_last3': 1.0,
        'tree_norm_last3': 2.6
    }
    shii_cols = []
    for crit, thresh in threshold_dict.items():
        shii_colname=f'shii_{crit}'
        df[shii_colname] = df[crit]>thresh
        shii_cols.append(shii_colname)
    
    df['shii_total'] = df[shii_cols].sum(axis=1)

    return df[['shii_total'] + shii_cols]
    

In [ ]:
shii_df = cdta_map.join(compute_shii(roll_df), how='right')

In [ ]:
cdta_grouped['HVI_RANK_ROUND'] = cdta_grouped['HVI_RANK'].round()

In [ ]:
# plasma_colors = [matplotlib.cm.plasma(i / 6) for i in range(6)]  # Samples at 0, 0.25, 0.5, 0.75, 1.0
plasma_colors = [matplotlib.cm.plasma(i / 5) for i in range(5)]  # Samples at 0, 0.25, 0.5, 0.75, 1.0
cmap_shii = matplotlib.colors.ListedColormap(plasma_colors)
cmap_hvi = matplotlib.colors.ListedColormap(['#eec131', '#eb9127', '#e65e1f', '#bc423b','#803a37'])
# Define boundaries for discrete bins (0 to 5)
norm= matplotlib.colors.BoundaryNorm(boundaries=[1, 1, 2, 3, 4, 5, 6], ncolors=5)
hvi_norm= matplotlib.colors.BoundaryNorm(boundaries=[1, 2, 3, 4, 5, 6], ncolors=5)
fig, axs = plt.subplots(1, 3, figsize=(12,3))
cdta_map.join(cdta_grouped).plot(
    column='HVI_RANK_ROUND', legend=False, vmin=1, vmax=5, ax = axs[0],
    cmap=cmap_hvi, norm=hvi_norm)
axs[0].set_title('Static HVI Score')
shii_df.reset_index().set_index(['date','cdta']).loc['2024-06-10'].plot(
    column='shii_total', legend=False, vmax=5, ax = axs[1],
    cmap=cmap_shii, norm=norm)
axs[1].set_title('Live Index, Not Heatwave\nJune 10, 2024')
shii_df.reset_index().set_index(['date','cdta']).loc['2024-06-21'].plot(
    column='shii_total', legend=False, vmax=5, ax = axs[2],
    cmap=cmap_shii, norm=norm)
axs[2].set_title('Live Index, Heatwave\nJune 21, 2024')

# Add custom colorbars with centered labels (midpoints of bins)
for i, ax in enumerate(axs):
    mappable = ax.collections[0]  # The plotted data
    if i == 0:
        cbar = fig.colorbar(mappable, ax=ax, shrink=0.8, ticks=[1.5, 2.5, 3.5, 4.5, 5.5])
        cbar.set_ticklabels(['1', '2', '3', '4', '5'])
        cbar.minorticks_off()
        # cbar.set_label('Score' if i == 0 else 'SHII Total')  # Optional labels
    else:
        cbar = fig.colorbar(mappable, ax=ax, shrink=0.8, ticks=[1.5, 2.5, 3.5, 4.5, 5.5])
        cbar.set_ticklabels(['0', '1', '2', '3', '4', '5'])
        cbar.minorticks_off()
        # cbar.set_label('Score' if i == 0 else 'SHII Total')  # Optional labels
    cdta_map.boundary.plot(ax=ax, color='grey',lw=0.1)
plt.tight_layout()


In [ ]:
# Source - https://stackoverflow.com/a/78609847
# Posted by Trenton McKinney, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-10, License - CC BY-SA 4.0

from matplotlib.animation import FuncAnimation, PillowWriter

# Initialize the plot with 2 rows
fig = plt.figure(figsize=(10, 7))
gs = fig.add_gridspec(2, 2, height_ratios=[1.5, 1], hspace=0.25, wspace=0.2)

# Top row: map animation
ax_map = fig.add_subplot(gs[0, :])
# Bottom row: two time series plots
ax_tmax = fig.add_subplot(gs[1, 0])
ax_ems = fig.add_subplot(gs[1, 1])

# Set fixed axis limits for map
xlim = (cdta_map.total_bounds[0], cdta_map.total_bounds[2])
ylim = (cdta_map.total_bounds[1], cdta_map.total_bounds[3])

ax_map.set_xlim(xlim)
ax_map.set_ylim(ylim)

# Initialize the colorbar in a separate axes to center the map
norm= matplotlib.colors.BoundaryNorm(boundaries=[0, 1, 2, 3, 4, 5, 6], ncolors=6)
sm = plt.cm.ScalarMappable(cmap=cmap_shii, norm=norm)
sm.set_array([])
cax = fig.add_axes([0.72, 0.53, 0.02, 0.3])  # Position for colorbar on the right
cbar = fig.colorbar(sm,cax=cax, ticks=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5])
cbar.set_ticklabels(['0', '1', '2', '3', '4', '5'])
cbar.minorticks_off()

# Date range for animation
start_date_anim = '2024-06-01'
end_date_anim = '2024-09-01'
dates = pd.date_range(start_date_anim, end_date_anim)

# Function to update the plot for each date
def animate(date):
    # Clear all axes
    ax_tmax.clear()
    ax_ems.clear()
    ax_map.clear()
    
    # Bottom left: tmax time series (full range)
    date_grouped['tmax'].loc[start_date_anim:end_date_anim].plot(
        ax=ax_tmax,
        color='darkred',
        ylabel='Temperature (C)',
        xlabel='',
        title='Daily Max Temperature',
        lw=2,
    )
    # Mark current date with vertical line
    ax_tmax.axvline(date, alpha=0.7, color='black', linestyle='--', lw=1.5, label='Current Date')
    ax_tmax.scatter([date], [date_grouped.loc[date, 'tmax']], color='black', s=50, zorder=5)
    ax_tmax.set_ylim([15, 40])
    
    # Bottom right: heat_ems_count_norm time series (full range)
    date_grouped['heat_ems_count'].loc[start_date_anim:end_date_anim].plot(
        ax=ax_ems,
        color='darkorange',
        ylabel='# Dispatches',
        xlabel='',
        title='Daily EMS Heat Dispatches',
        lw=2
    )
    # Mark current date with vertical line
    ax_ems.axvline(date, alpha=0.7, color='black', linestyle='--', lw=1.5, label='Current Date')
    ax_ems.scatter([date], [date_grouped.loc[date, 'heat_ems_count_norm']], color='black', s=40, zorder=5)
    
    # Top: map of current data
    ax_map.set_xlim(xlim)
    ax_map.set_ylim(ylim)
    boundary = cdta_map.boundary.plot(ax=ax_map, edgecolor='black', linewidth=0.3)
    
    # Plot the data for the current date
    shii_df.reset_index().set_index(['date','cdta']).loc[date].plot(
        column='shii_total', legend=False, vmax=5, ax=ax_map,
        cmap=cmap_shii, norm=norm)
    
    # Add date annotation at the top
    fig.suptitle(f'311 Heat Impact Index\n{date.date()}', fontsize=14, fontweight='bold')

# Create the animation
animation = FuncAnimation(fig, animate, frames=dates, repeat=False, interval=100)

# Save the animation as a GIF
writer = PillowWriter(fps=1)
animation.save('test_shii.gif', writer=writer, dpi=200)

# Show the plot
plt.show()

In [ ]:
# Source - https://stackoverflow.com/a/78609847
# Posted by Trenton McKinney, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-10, License - CC BY-SA 4.0

from matplotlib.animation import FuncAnimation, PillowWriter

# Initialize the plot
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 8))

# Set fixed axis limits
xlim = (cdta_map.total_bounds[0], cdta_map.total_bounds[2])
ylim = (cdta_map.total_bounds[1], cdta_map.total_bounds[3])

ax.set_xlim(xlim)
ax.set_ylim(ylim)
# Plot initial boundaries
boundary = cdta_map.boundary.plot(ax=ax, edgecolor='black', linewidth=0.3)

# Initialize the colorbar variable with a fixed normalization
norm = plt.Normalize(vmin=0, vmax=5)
sm = plt.cm.ScalarMappable(cmap=cmap_shii, norm=norm)
sm.set_array([])  # Only needed for adding the colorbar
cbar = fig.colorbar(sm, ax=ax, shrink=0.8, ticks=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5])
cbar.set_ticklabels(['0', '1', '2', '3', '4', '5'])
cbar.minorticks_off()

# Function to update the plot for each year
def animate(date):
    ax.clear()
    # Set the fixed axis limits
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    # Plot initial boundaries
    boundary = cdta_map.boundary.plot(ax=ax, edgecolor='black', linewidth=0.3)
    
    # Plot the data for the current year
    shii_df.reset_index().set_index(['date','cdta']).loc[date].plot(
        column='shii_total', legend=False, vmax=5, ax = ax,
        cmap=cmap_shii, norm=norm)
 

    # Add year annotation at the top
    ax.annotate(f'311 Heat Impact Index Evolution\n{date.date()}', xy=(0.5, 1.05), xycoords='axes fraction', fontsize=12, ha='center')

# Create the animation
dates = pd.date_range('2024-07-01', '2024-07-15')
animation = FuncAnimation(fig, animate, frames=dates, repeat=False, interval=20)

# Save the animation as a GIF
writer = PillowWriter(fps=1)
animation.save('test_shii_only.gif', writer=writer)

# Show the plot
plt.show()


In [ ]:

# HVI Rank vs our other predictors
fig, axs = plt.subplots(2,3, figsize=(12, 8))
color_list = ['#D62828', '#FCBF49', 'purple', '#2A9D8F']
cdta_grouped_sum.plot.scatter(
    x='heat_ems_count_norm',
    y='HVI_RANK',
    ax=axs[0, 0],
    color='darkorange',
    title='Heat EMS Dispatches',
    ylabel='HVI Score (1-5)',
    xlabel='# EMS Dispatches (per 100k)',
    )
cdta_grouped_sum.plot.scatter(
    x='hydrant_all_norm',
    y='HVI_RANK',
    ax=axs[0, 1],
    color=color_list[0],
    title='Hydrant 311',
    ylabel='HVI Score (1-5)',
    xlabel='# 311 Requests (per 100k)' 
    )
cdta_grouped_sum.plot.scatter(
    x='ventilation_norm',
    y='HVI_RANK',
    ax=axs[0,2],
    color=color_list[1],
    title='Ventilation 311',
    ylabel='HVI Score (1-5)',
    xlabel='# 311 Requests (per 100k)' 
    )
cdta_grouped_sum.plot.scatter(
    x='power_norm',
    y='HVI_RANK',
    ax=axs[1, 0],
    color=color_list[2],
    title='Power 311',
    ylabel='HVI Score (1-5)',
    xlabel='# 311 Requests (per 100k)',
    )
cdta_grouped_sum.plot.scatter(
    x='tree_norm',
    y='HVI_RANK',
    ax=axs[1, 1],
    color='green',
    title='New Tree 311',
    ylabel='HVI Score (1-5)',
    xlabel='# 311 Requests (per 100k)' 
    )
cdta_grouped_sum.plot.scatter(
    x='tree_norm',
    y='HVI_RANK',
    ax=axs[1, 1],
    color='green',
    title='New Tree 311',
    ylabel='HVI Score (1-5)',
    xlabel='# 311 Requests (per 100k)' 
    )
cdta_map.join(cdta_grouped).plot(
    column='HVI_RANK_ROUND', legend=False, vmin=1, vmax=5, ax = axs[1,2],
    cmap=cmap_hvi, norm=hvi_norm)
axs[1,2].set_title('HVI')
fig.tight_layout()

In [ ]:
# Fig 3: Per CDTA comparisons with heat ems counts
fig, axs = plt.subplots(3,3, figsize=(12, 8))
color_list = ['#D62828', '#FCBF49', 'purple', '#2A9D8F']
cdta_grouped_sum.plot.scatter(
    x='HVI_RANK',
    y='heat_ems_count',
    ax=axs[0, 0],
    color='darkorange',
    title='HVI Score',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='HVI Score (1-5)',
    )
cdta_grouped_sum.plot.scatter(
    x='MEDIAN_INCOME',
    y='heat_ems_count',
    ax=axs[1, 0],
    color='green',
    title='Median Income',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='Median Income ($/year)',
    )
cdta_grouped_sum.plot.scatter(
    x='GREENSPACE',
    y='heat_ems_count',
    ax=axs[2,0],
    color='darkgreen',
    title='Greenspace',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='Greenspace (%)',
    )
cdta_grouped_sum.plot.scatter(
    x='SURFACE_TEMP',
    y='heat_ems_count',
    ax=axs[0, 1],
    color='purple',
    title='Surface Temp',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='Surface Temp (F)',
    )
cdta_grouped_sum.plot.scatter(
    x='PCT_BLACK_POP',
    y='heat_ems_count',
    ax=axs[1, 1],
    color='darkblue',
    title='Black population %',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='% Black population'
    )
cdta_grouped_sum.plot.scatter(
    x='PCT_HOUSEHOLDS_AC',
    y='heat_ems_count',
    ax=axs[2, 1],
    color=color_list[3],
    title='Household AC %',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='% Households with AC'
    )
cdta_grouped_sum.plot.scatter(
    x='hydrant_all',
    y='heat_ems_count',
    ax=axs[0, 2],
    color=color_list[0],
    title='311 Hydrant',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='# Rqests',
    )
cdta_grouped_sum.plot.scatter(
    x='ventilation',
    y='heat_ems_count',
    ax=axs[1, 2],
    color=color_list[1],
    title='311 Ventilation',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='# Requests'
    )
cdta_grouped_sum.plot.scatter(
    x='power',
    y='heat_ems_count',
    ax=axs[2, 2],
    color=color_list[2],
    title='311 Power',
    ylabel='Heat Illness EMS Dispatches',
    xlabel='# Requests',
    )
fig.tight_layout()